# 05 - Open WebUI End-to-End Flow

This notebook validates the full path:
**Browser -> Open WebUI -> langchain-demo proxy -> Ollama -> LangSmith**

You will trigger prompts from the browser, then verify traces and metrics here.


In [ ]:
import os
import sys
from pathlib import Path

expected = "/usr/local/bin/python3.11"
print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version.split()[0]}")
if Path(sys.executable).resolve().as_posix() != expected:
    print(f"WARNING: This notebook is designed for {expected}.")
    print("Switch the notebook kernel to Python 3.11 before continuing.")
else:
    print("Kernel check passed.")


In [ ]:
import importlib
import subprocess
import sys

packages = ["requests", "pandas", "matplotlib", "pyyaml", "langsmith", "tabulate"]
missing = []
for pkg in packages:
    module_name = "yaml" if pkg == "pyyaml" else pkg
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pkg)

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("All common packages are already installed.")


In [ ]:
import os
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import pandas as pd
import requests
from langsmith import Client

LANGSMITH_ENDPOINT = os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "k3s-ollama-gemma-local")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")
OPEN_WEBUI_URL = "http://localhost:8080"


def find_nested_values(obj, wanted_keys: set[str]) -> list[tuple[str, object]]:
    found = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key in wanted_keys:
                found.append((key, value))
            found.extend(find_nested_values(value, wanted_keys))
    elif isinstance(obj, (list, tuple)):
        for item in obj:
            found.extend(find_nested_values(item, wanted_keys))
    return found


def first_numeric(candidates: list[tuple[str, object]]) -> tuple[float | None, str | None]:
    for key, value in candidates:
        if isinstance(value, bool):
            continue
        if isinstance(value, (int, float)):
            return float(value), key
        if isinstance(value, str):
            try:
                return float(value), key
            except ValueError:
                continue
    return None, None


def extract_run_metrics(run) -> dict[str, object]:
    outputs = getattr(run, "outputs", {}) or {}
    status_code, status_source = first_numeric(find_nested_values(outputs, {"status_code", "http_status", "response_status"}))
    latency_ms, latency_source = first_numeric(find_nested_values(outputs, {"latency_ms", "duration_ms", "elapsed_ms"}))
    if latency_ms is None:
        start_time = getattr(run, "start_time", None)
        end_time = getattr(run, "end_time", None)
        if start_time is not None and end_time is not None:
            latency_ms = round((end_time - start_time).total_seconds() * 1000, 2)
            latency_source = "timestamps"
    if status_code is not None:
        status_code = int(status_code)
    return {
        "status_code": status_code,
        "status_source": status_source,
        "latency_ms": latency_ms,
        "latency_source": latency_source,
    }


client = None
if not LANGSMITH_API_KEY:
    print("LANGSMITH_API_KEY/LANGCHAIN_API_KEY is not set. LangSmith cells will run in skip mode.")
else:
    try:
        client = Client(api_url=LANGSMITH_ENDPOINT, api_key=LANGSMITH_API_KEY)
    except Exception as exc:
        print(f"Unable to initialize LangSmith client: {exc}")

print("LangSmith endpoint:", LANGSMITH_ENDPOINT)
print("LangSmith project :", LANGSMITH_PROJECT)
print("LANGSMITH_READY   :", client is not None)


## Validate Open WebUI Endpoints


In [ ]:
for path in ["/", "/health", "/api/version"]:
    url = f"{OPEN_WEBUI_URL}{path}"
    try:
        r = requests.get(url, timeout=8)
        print(url, "->", r.status_code)
        if "application/json" in r.headers.get("content-type", ""):
            print(r.json())
    except Exception as exc:
        print(url, "->", exc)


## Capture Baseline Run Count


In [ ]:
baseline_time = datetime.now(timezone.utc)
baseline_runs = []
if client is None:
    print("Skipping baseline run fetch: LangSmith client is not configured.")
else:
    try:
        baseline_runs = list(client.list_runs(project_name=LANGSMITH_PROJECT, start_time=baseline_time - timedelta(minutes=15), limit=200))
    except Exception as exc:
        print(f"Baseline run fetch failed: {exc}")

print("Baseline UTC:", baseline_time.isoformat())
print("Recent runs before manual browser chat:", len(baseline_runs))


## Manual Browser Step
1. Open `http://localhost:8080/`
2. Send 3-5 prompts from Open WebUI chat
3. Return here and run the next cell


In [ ]:
new_runs = []
if client is None:
    print("Skipping new run fetch: LangSmith client is not configured.")
else:
    try:
        new_runs = list(client.list_runs(project_name=LANGSMITH_PROJECT, start_time=baseline_time, limit=300))
    except Exception as exc:
        print(f"New run fetch failed: {exc}")

print("Runs observed after baseline:", len(new_runs))

rows = []
for run in new_runs:
    tags = getattr(run, "tags", []) or []
    metrics = extract_run_metrics(run)
    rows.append(
        {
            "run_id": str(getattr(run, "id", "")),
            "name": getattr(run, "name", ""),
            "start_time": getattr(run, "start_time", None),
            "error": getattr(run, "error", None),
            "status_code": metrics["status_code"],
            "status_source": metrics["status_source"],
            "latency_ms": metrics["latency_ms"],
            "latency_source": metrics["latency_source"],
            "tags": tags,
            "has_open_webui_tag": "open-webui" in tags,
            "has_proxy_tag": "ollama-proxy" in tags,
        }
    )

cols = [
    "run_id",
    "name",
    "start_time",
    "error",
    "status_code",
    "status_source",
    "latency_ms",
    "latency_source",
    "tags",
    "has_open_webui_tag",
    "has_proxy_tag",
]
webui_df = pd.DataFrame(rows, columns=cols)
webui_df.head(30)


In [ ]:
if not webui_df.empty:
    filtered = webui_df[(webui_df["has_open_webui_tag"]) | (webui_df["has_proxy_tag"])].copy()
    filtered["latency_ms"] = pd.to_numeric(filtered["latency_ms"], errors="coerce")
    filtered["start_time"] = pd.to_datetime(filtered["start_time"], errors="coerce")

    print("Tagged Open WebUI/Proxy runs:", len(filtered))
    print("Rows with numeric latency:", int(filtered["latency_ms"].notna().sum()))
    print("Latency source counts:")
    display(filtered["latency_source"].fillna("missing").value_counts().rename_axis("latency_source").to_frame("runs"))
    display(filtered[["start_time", "name", "status_code", "status_source", "latency_ms", "latency_source", "error"]].tail(20))

    if not filtered.empty:
        by_min = (
            filtered.set_index("start_time")
            .resample("1min")
            .size()
            .rename("runs_per_min")
            .reset_index()
        )
        ax = by_min.plot(x="start_time", y="runs_per_min", figsize=(10, 4), marker="o", title="Open WebUI Throughput (runs/min)")
        ax.set_ylabel("Runs / minute")
        plt.show()

        lat = filtered[filtered["latency_ms"].notna()].tail(50)
        if not lat.empty:
            ax = lat.plot(x="start_time", y="latency_ms", figsize=(10, 4), marker="o", title="Open WebUI Flow Latency")
            ax.set_ylabel("Latency (ms)")
            plt.show()
        else:
            print("No numeric latency available. The notebook falls back to run timestamps when LangSmith outputs omit latency_ms.")


## Confirm Runtime Chain in Kubernetes


In [ ]:
import subprocess

for cmd in [
    "kubectl get pods -n llm-observability -l app.kubernetes.io/name=langchain-demo",
    "kubectl get pods -n llm-observability -l app.kubernetes.io/name=ollama",
    "kubectl get pods -n llm-observability -l app.kubernetes.io/name=open-webui",
    "kubectl logs -n llm-observability deploy/langchain-demo --tail=60",
]:
    print("=" * 120)
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(p.stdout if p.stdout else p.stderr)


## Done
You now have proof that browser-triggered chats are visible in LangSmith and can be charted programmatically.
